# 12 — Solution Progression Embeddings

**Phase 5 E-3** — Create low-dimensional (32-D) embeddings of learner solutions using
`TruncatedSVD` applied to behavioral complexity features. These act as a behavioral proxy
for semantic solution embeddings when raw SQL text is unavailable.

**Inputs:**
- `data/features/sql_complexity_v1.parquet` — NB10 complexity features (required)
- `data/features/error_clusters_v1.parquet` — NB11 cluster features (optional, merged if present)

**Output:**
- `data/features/solution_embeddings_v1.npz` — 32-D embeddings per learner×task
- `data/features/embeddings_manifest_v1.json` — schema/stats manifest

Research constraint: `label_validity = pilot_only` — behavioral proxy only (no SQL text available).

In [ ]:
# cfg-01
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

SCHEMA_VERSION  = 'embeddings_v1'
N_COMPONENTS    = 32
RANDOM_STATE    = 42

FEAT_DIR        = Path('data/features')
FEAT_DIR.mkdir(parents=True, exist_ok=True)

# Prereq paths
COMPLEXITY_PATH = FEAT_DIR / 'sql_complexity_v1.parquet'
CLUSTER_PATH    = FEAT_DIR / 'error_clusters_v1.parquet'

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    h.update(p.read_bytes())
    return h.hexdigest()[:16]

print(f'SCHEMA_VERSION : {SCHEMA_VERSION}')
print(f'N_COMPONENTS   : {N_COMPONENTS}')
print(f'Complexity     : {COMPLEXITY_PATH} (exists={COMPLEXITY_PATH.exists()})')
print(f'Clusters       : {CLUSTER_PATH}    (exists={CLUSTER_PATH.exists()})')

## 2. Load and merge feature tables

In [ ]:
# load-01
if not COMPLEXITY_PATH.exists():
    raise FileNotFoundError(
        f'{COMPLEXITY_PATH} not found — run NB10 first.'
    )

df_comp = pd.read_parquet(COMPLEXITY_PATH)
print(f'Complexity rows: {len(df_comp):,}  columns: {list(df_comp.columns)}')

# Optional: merge error clusters
if CLUSTER_PATH.exists():
    df_cl = pd.read_parquet(CLUSTER_PATH)
    # Identify join keys (may vary depending on NB10/NB11 outputs)
    join_keys = [k for k in ['academy_member_id', 'task_id'] if k in df_comp.columns and k in df_cl.columns]
    if join_keys:
        df_comp = df_comp.merge(
            df_cl[join_keys + [c for c in df_cl.columns if c not in join_keys and c != 'error_type']],
            on=join_keys, how='left'
        )
        print(f'After cluster merge: {len(df_comp):,} rows, {len(df_comp.columns)} columns')
    else:
        print('[INFO] Cluster parquet found but no join keys matched — skipping merge')
else:
    print('[INFO] error_clusters_v1.parquet not found — proceeding with complexity features only')

print(df_comp.dtypes)

## 3. Build feature matrix and compute TruncatedSVD embeddings

In [ ]:
# embed-01
# Identify numeric feature columns (exclude ID/key columns)
NON_FEATURE_COLS = {
    'academy_member_id', 'task_id', 'batch_id', 'batch_code',
    'profile_id', 'participant_code',
    'dominant_cluster_terms', 'error_type',
    'schema_version', 'created_at', 'created_at_utc',
}

feature_cols = [
    c for c in df_comp.columns
    if c not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(df_comp[c])
]
print(f'Feature columns ({len(feature_cols)}): {feature_cols}')

if len(feature_cols) == 0:
    raise RuntimeError('No numeric feature columns found — check NB10 output schema.')

X_raw = df_comp[feature_cols].fillna(0).values.astype(np.float32)
print(f'Feature matrix shape: {X_raw.shape}')

# Standardise
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# TruncatedSVD — like PCA but works on dense + can be applied to sparse
n_comp = min(N_COMPONENTS, X_scaled.shape[1], X_scaled.shape[0] - 1)
svd    = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
X_emb  = svd.fit_transform(X_scaled).astype(np.float32)

explained_var_ratio = svd.explained_variance_ratio_
cumulative_var      = float(explained_var_ratio.cumsum()[-1])

print(f'Embedding shape     : {X_emb.shape}')
print(f'Explained variance  : {cumulative_var:.1%} ({n_comp} components)')
print(f'Top-3 component var : {explained_var_ratio[:3].tolist()}')

## 4. Compute pairwise cosine similarity (within-task)

In [ ]:
# cosine-01
# Mean within-task cosine similarity — proxy for solution convergence
within_task_sim: dict[str, float] = {}

if 'task_id' in df_comp.columns:
    for task_id, grp in df_comp.groupby('task_id'):
        idx = grp.index.tolist()
        if len(idx) < 2:
            continue
        X_task = X_emb[idx]
        sim_mat = cosine_similarity(X_task)
        # Mean of upper triangle (exclude diagonal)
        n = sim_mat.shape[0]
        upper = sim_mat[np.triu_indices(n, k=1)]
        within_task_sim[str(task_id)] = float(upper.mean())

    if within_task_sim:
        vals = list(within_task_sim.values())
        print(f'Within-task cosine similarity:')
        print(f'  tasks analysed : {len(within_task_sim)}')
        print(f'  mean           : {np.mean(vals):.4f}')
        print(f'  min/max        : {min(vals):.4f} / {max(vals):.4f}')
        for task, sim in sorted(within_task_sim.items(), key=lambda x: x[1]):
            print(f'  {task}: {sim:.4f}')
    else:
        print('[INFO] Not enough learners per task for within-task similarity')
else:
    print('[INFO] task_id column not found — skipping within-task similarity')

In [ ]:
# viz-01
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPORTS_DIR = Path('reports/phase4')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Scree plot: explained variance per SVD component
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree
ax0 = axes[0]
ax0.bar(range(1, n_comp + 1), explained_var_ratio * 100, color='#F37021', edgecolor='white')
ax0.set_title('SVD Scree Plot (NB12)', fontsize=11, fontweight='bold')
ax0.set_xlabel('Component')
ax0.set_ylabel('Explained Variance (%)')
ax0.set_xlim(0.5, n_comp + 0.5)

# Cumulative
ax1 = axes[1]
ax1.plot(range(1, n_comp + 1), explained_var_ratio.cumsum() * 100,
         color='#F37021', marker='o', markersize=4, linewidth=2)
ax1.axhline(y=80, color='#64748B', linestyle='--', linewidth=1, alpha=0.7)
ax1.set_title('Cumulative Variance (NB12)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Components')
ax1.set_ylabel('Cumulative Variance (%)')
ax1.set_ylim(0, 105)

fig.tight_layout()
out_png = REPORTS_DIR / 'nb12_svd_variance.png'
fig.savefig(out_png, dpi=150)
plt.close(fig)
print(f'Plot saved: {out_png}')

## 5. Save artifacts

In [ ]:
# artifacts-01
EMBEDDINGS_PATH = FEAT_DIR / 'solution_embeddings_v1.npz'
MANIFEST_PATH   = FEAT_DIR / 'embeddings_manifest_v1.json'

# Build index arrays for the npz
member_ids = df_comp['academy_member_id'].values if 'academy_member_id' in df_comp.columns else np.arange(len(df_comp))
task_ids   = df_comp['task_id'].values if 'task_id' in df_comp.columns else np.zeros(len(df_comp), dtype=int)

np.savez_compressed(
    EMBEDDINGS_PATH,
    embeddings       = X_emb,
    academy_member_id= member_ids,
    task_id          = task_ids,
    feature_cols     = np.array(feature_cols),
    components       = svd.components_.astype(np.float32),
    explained_variance_ratio = explained_var_ratio.astype(np.float32),
)

comp_sha = sha256_file(COMPLEXITY_PATH)

manifest = {
    'schema_version': SCHEMA_VERSION,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_files': {
        'complexity_parquet': str(COMPLEXITY_PATH),
        'complexity_sha'    : comp_sha,
        'cluster_parquet'   : str(CLUSTER_PATH) if CLUSTER_PATH.exists() else None,
        'cluster_sha'       : sha256_file(CLUSTER_PATH) if CLUSTER_PATH.exists() else None,
    },
    'parameters': {
        'algorithm'     : 'TruncatedSVD',
        'n_components'  : int(n_comp),
        'random_state'  : RANDOM_STATE,
        'scaler'        : 'StandardScaler',
        'feature_cols'  : feature_cols,
    },
    'dataset_stats': {
        'n_learner_task_rows'   : int(len(df_comp)),
        'n_features_input'      : len(feature_cols),
        'n_components_actual'   : int(n_comp),
        'cumulative_var_pct'    : round(cumulative_var * 100, 2),
        'top3_var_pct'          : [round(v * 100, 2) for v in explained_var_ratio[:3].tolist()],
        'within_task_similarity': within_task_sim,
    },
    'embedding_note': (
        'Behavioral proxy embeddings (no SQL text available). '
        'TruncatedSVD on NB10 complexity features serves as surrogate for semantic embeddings.'
    ),
    'data_warning': 'PILOT ONLY — proxy_behavioral labels, label_validity=pilot_only.',
    'artifacts': {
        'solution_embeddings': str(EMBEDDINGS_PATH),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=str))

print(f'solution_embeddings_v1.npz  : {EMBEDDINGS_PATH.stat().st_size:,} bytes')
print(f'embeddings_manifest_v1.json : {MANIFEST_PATH.stat().st_size:,} bytes')

## 6. Validation

In [ ]:
# validate-01
checks = []

def chk(name: str, passed: bool, detail: str = '') -> None:
    icon = '[OK]' if passed else '[FAIL]'
    checks.append({'name': name, 'passed': passed, 'detail': detail})
    print(f'{icon}  {name}' + (f' — {detail}' if detail else ''))

# Load back to verify
loaded = np.load(EMBEDDINGS_PATH, allow_pickle=True)

chk('npz exists',              EMBEDDINGS_PATH.exists(),               str(EMBEDDINGS_PATH))
chk('embeddings key present',  'embeddings' in loaded,                 str(list(loaded.keys())))
chk('embedding shape correct', loaded['embeddings'].shape == (len(df_comp), n_comp),
    f'{loaded["embeddings"].shape}')
chk('no NaN in embeddings',    not np.isnan(loaded['embeddings']).any(), '')
chk('manifest exists',         MANIFEST_PATH.exists(),                 str(MANIFEST_PATH))
chk('n_components >= 1',       n_comp >= 1,                           f'n_comp={n_comp}')
chk('cumulative var > 0',      cumulative_var > 0,                    f'{cumulative_var:.1%}')

n_pass  = sum(c['passed'] for c in checks)
n_total = len(checks)
print(f'\nValidation: {n_pass}/{n_total} checks passed')

failed = [c['name'] for c in checks if not c['passed']]
if failed:
    raise RuntimeError(f'NB12 validation failed: {failed}')

print('NB12 COMPLETE — Phase 5 E notebooks done (NB10 + NB11 + NB12).')